# 📡 Automated Anomaly Detection & Business Monitoring

## Executive Summary
Daily revenue averages $10,000 — but on one Tuesday it drops to $2,000. Is it a data error, a system failure, or a real business problem? This notebook implements a 5-stage automated monitoring pipeline:
1. **Threshold-based rules** for hard-limit violations
2. **Z-score statistical detection** for subtle deviations
3. **Severity classification** to avoid alert fatigue
4. **Audit logging** for investigation tracking
5. **Time-series visualization** with anomaly flags

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

sys.path.append(os.path.abspath('..'))
from scripts.anomaly_detection_assignment import (
    generate_revenue_dataset,
    detect_anomalies_zscore,
    classify_severity
)

df = generate_revenue_dataset(n_days=60)
print(f'Dataset loaded: {len(df)} days')
df.head()

### Task 1: Threshold-Based Anomaly Detection (1 mark)
Define business rules and check today's metrics against them.

In [ ]:
alert_rules = {
    'daily_revenue':     {'min': 5000,  'max': 50000},
    'transaction_count': {'min': 100,   'max': 10000},
    'signup_rate':       {'min': 10,    'max': 500}
}

def check_thresholds(metrics, rules):
    alerts = []
    for metric_name, rule in rules.items():
        value = metrics[metric_name]
        if value < rule['min']:
            alerts.append({'metric': metric_name, 'value': value, 'threshold': rule['min'], 'direction': 'BELOW_MIN', 'severity': 'HIGH'})
        elif value > rule['max']:
            alerts.append({'metric': metric_name, 'value': value, 'threshold': rule['max'], 'direction': 'ABOVE_MAX', 'severity': 'MEDIUM'})
    return alerts

# Simulate today = anomaly day (Day 15 — revenue drop)
anomaly_row = df.iloc[15]
today_metrics = {
    'daily_revenue':     float(anomaly_row['amount']),
    'transaction_count': int(anomaly_row['transaction_count']),
    'signup_rate':       int(anomaly_row['signup_rate'])
}

alerts = check_thresholds(today_metrics, alert_rules)
for alert in alerts:
    print(f"⚠️ {alert['metric']} {alert['direction']}: {alert['value']} (threshold: {alert['threshold']}, severity: {alert['severity']})")

### Task 2: Statistical Anomaly Detection with Z-Score (1 mark)
Detect values beyond 2 standard deviations over a 30-day lookback window.

In [ ]:
daily_revenue = df.set_index('date')['amount'].tail(30)
anomalies, z_scores = detect_anomalies_zscore(daily_revenue, threshold=2)

print(f'Detected {len(anomalies)} anomalies out of {len(daily_revenue)} days')
for date, value in anomalies.items():
    print(f'  {date}: ${value:.0f} (z-score: {z_scores[date]:.2f})')

### Task 3: Severity Classification (1 mark)
Classify each anomaly into CRITICAL / HIGH / MEDIUM / LOW using z-score thresholds.

In [ ]:
rev_mean = daily_revenue.mean()
rev_std  = daily_revenue.std()

anomaly_severity = []
for date, value in anomalies.items():
    severity = classify_severity(value, rev_mean, rev_std)
    anomaly_severity.append({'date': date, 'value': value, 'z_score': round(float(z_scores[date]),3), 'severity': severity})

severity_df = pd.DataFrame(anomaly_severity)
display(severity_df)

critical = severity_df[severity_df['severity'].isin(['CRITICAL', 'HIGH'])]
print(f"\n⚠️ {len(critical)} critical anomalies require investigation")

### Task 4: Anomaly Logging and Audit Trail (1 mark)
Persist all anomalies to CSV with timestamps and investigation status.

In [ ]:
expected_low  = rev_mean - 2 * rev_std
expected_high = rev_mean + 2 * rev_std

anomaly_log = []
for date, value in anomalies.items():
    severity = classify_severity(value, rev_mean, rev_std)
    anomaly_log.append({
        'logged_at':      pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S'),
        'anomaly_date':   str(date),
        'metric':         'daily_revenue',
        'actual_value':   round(value, 2),
        'expected_range': f'${expected_low:,.0f} – ${expected_high:,.0f}',
        'z_score':        round(float(z_scores[date]), 3),
        'severity':       severity,
        'status':         'OPEN'
    })

anomalies_df = pd.DataFrame(anomaly_log)
anomalies_df.to_csv('anomalies_log.csv', index=False)
print(f'Logged {len(anomalies_df)} anomalies to anomalies_log.csv')
display(anomalies_df)

### Task 5: Visualization with Flagged Points (1 mark)
Plot time-series with rolling average, ±2σ band, and annotated anomaly markers.

In [ ]:
dates_plot  = [pd.Timestamp(d) for d in daily_revenue.index]
rolling_avg = daily_revenue.rolling(window=7).mean()

fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(dates_plot, daily_revenue.values, marker='o', label='Daily Revenue', linewidth=2, color='#3b82f6')
ax.plot(dates_plot, rolling_avg.values, label='7-day MA', color='green', linewidth=2, linestyle='--')

mean = daily_revenue.mean()
std  = daily_revenue.std()
ax.fill_between(dates_plot, mean-2*std, mean+2*std, alpha=0.2, color='blue', label='Expected Range ±2σ')

for date, value in anomalies.items():
    ax.scatter(pd.Timestamp(date), value, color='red', s=200, marker='X', zorder=5)
    ax.annotate('ANOMALY', (pd.Timestamp(date), value),
                xytext=(0, 10), textcoords='offset points', ha='center', fontweight='bold', color='red')

ax.set_xlabel('Date', fontsize=12)
ax.set_ylabel('Revenue ($)', fontsize=12)
ax.set_title('Daily Revenue with Anomalies Flagged', fontsize=14)
ax.legend()
ax.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('anomaly_detection.png', dpi=150)
plt.show()
print('Funnel visualization saved as anomaly_detection.png')